In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision.utils as vutils
from tqdm import tqdm

# --------------------
# CONFIG
# --------------------
latent_dim = 128
batch_size = 128
epochs = 100
lr = 1e-3
skip_dropout_prob = 0.3
device = torch.device("mps")

### Autoencoder Architecture
- Encoder
- Decoder

Picture > Encoder > Latent > Decoder > Picture

**Variational Autoencoder**

Picture > Encoder > Mean and Log-Variance > Decoder > Picture



### Dataset
- We simply take some subset of the coin images and for now don't care about the labels.
- We also resize the images to 32x32 for computational reasons.

Data is structured as follows:

data/
    coins/
        all/
            randomname.png
            randomname2.png
            ...

In [4]:
# --------------------
# DATASET
# --------------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),  # Converts to [0, 1]
])

dataset = datasets.ImageFolder(
    root='data/coins',
    transform=transform,
    target_transform=None
)

train_loader = DataLoader(
    dataset=dataset,
    batch_size=batch_size,
    shuffle=True
)

### UNet defintion
- UNet is a basic encoder-decoder architecture (https://www.researchgate.net/publication/359148875/figure/fig1/AS:1132274238144513@1646966623381/Structure-diagram-of-UNet.png)
- The encoder is a series of convolutional layers that reduce the spatial dimensions of the input image.
- The decoder is a series of transposed convolutional layers that increase the spatial dimensions of the input image.

**We don't predict a latent directly**. Instead we predict a **mean** and a **log variance** for the latent and then sample from that distribution in the decoder.

In [5]:
class UNetVAE(nn.Module):
    def __init__(self, latent_dim=128, skip_dropout_prob=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.skip_dropout_prob = skip_dropout_prob

        # -------- ENCODER --------
        self.enc1 = nn.Sequential(nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU())    # 32x32 → 16x16
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU())  # 16x16 → 8x8
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU()) # 8x8 → 4x4
        self.enc4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.ReLU()) # 4x4 → 2x2

        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(512 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(512 * 2 * 2, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 512 * 2 * 2)

        # -------- DECODER --------
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(512 + 256, 256, 4, 2, 1), nn.ReLU())  # 2x2 → 4x4
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(256 + 128, 128, 4, 2, 1), nn.ReLU())  # 4x4 → 8x8
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(128 + 64, 64, 4, 2, 1), nn.ReLU())    # 8x8 → 16x16
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Sigmoid())        # 16x16 → 32x32

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def encode(self, x):
        s1 = self.enc1(x)  # [B, 64, 16, 16]
        s2 = self.enc2(s1) # [B, 128, 8, 8]
        s3 = self.enc3(s2) # [B, 256, 4, 4]
        x = self.enc4(s3)  # [B, 512, 2, 2]
        flat = self.flatten(x)
        mu = self.fc_mu(flat)
        logvar = self.fc_logvar(flat)
        return mu, logvar, (s1, s2, s3)

    def decode(self, z, skips=None):
        x = self.fc_decode(z).view(-1, 512, 2, 2)

        if skips is None:
            skips = (
                torch.zeros(x.size(0), 64, 16, 16, device=x.device),
                torch.zeros(x.size(0), 128, 8, 8, device=x.device),
                torch.zeros(x.size(0), 256, 4, 4, device=x.device)
            )

        s1, s2, s3 = skips

        s3 = F.interpolate(s3, size=x.shape[2:], mode='nearest') # ConvTranspose2d
        x = self.dec4(torch.cat([x, s3], dim=1))

        s2 = F.interpolate(s2, size=x.shape[2:], mode='nearest')
        x = self.dec3(torch.cat([x, s2], dim=1))

        s1 = F.interpolate(s1, size=x.shape[2:], mode='nearest')
        x = self.dec2(torch.cat([x, s1], dim=1))

        x = self.dec1(x)
        return x


    def forward(self, x):
        mu, logvar, skips = self.encode(x)
        z = self.reparameterize(mu, logvar)
        if self.training and self.skip_dropout_prob > 0.0:
            skips = [s if torch.rand(1).item() > self.skip_dropout_prob else torch.zeros_like(s) for s in skips]
        x_hat = self.decode(z, skips)
        return x_hat, mu, logvar

### Loss Function

In [6]:
def vae_loss(x_hat, x, mu, logvar):
    recon = F.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) # KL divergence between q(z|x) and p(z)
    # Kullback-Leibler divergence: We make sure that the latents stay in a normal distribution and do not wander outside of it
    return recon + kl

### Model Training

- We initialize the model, the optimizer and a scheduler so that the learning rate is reduced if we are not making progress.
- The scheduler is a one cycle scheduler with warmup https://www.researchgate.net/publication/385152191/figure/fig5/AS:11431281285250637@1729671781353/Graph-of-Learning-Rate-in-OneCycleLR-scheduler.ppm

In [8]:
model = UNetVAE(skip_dropout_prob=skip_dropout_prob).to(device)
model.load_state_dict(torch.load('model.pth'))

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=lr,
    epochs=epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    div_factor=25,
    final_div_factor=1e4
)

### Training Loop
- We train the model for a fixed number of epochs
- We save the generated samples every 10 epochs

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}], LR [{scheduler.get_last_lr()[0]:.6f}]", leave=False)

    for x, _ in loop:
        x = x.to(device)
        x_hat, mu, logvar = model(x)
        loss = vae_loss(x_hat, x, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item() / x.size(0))

    scheduler.step()
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} | Average Loss: {avg_loss:.2f}")

    # Optional: save generated samples every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            z = torch.randn(16, latent_dim).to(device)
            samples = model.decode(z, skips=None)
            vutils.save_image(samples, f"samples_epoch_{epoch+1}.png", nrow=4)

In [17]:
model.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim).to(device) * 0.5
    samples = model.decode(z, skips=None)
    vutils.save_image(samples, f"samples_test.png", nrow=4)  